# sebal-et-gee — Day 1: AOI Setup + First Landsat Preview

**Goal:** Load Thrace boundary, overlay 8 precipitation stations, select a clear Landsat 9 scene, and visualize it on an interactive map.

**Output by end of notebook:**
- An interactive map showing: Thrace boundary, 8 stations, Landsat 9 RGB + LST preview
- A `thrace_boundary.geojson` file ready for the pipeline
- A `thrace_stations.csv` file with the 8 stations used in the Scientific Reports manuscript

**Run order:** Execute cells top-to-bottom. The first cell installs dependencies (once per Colab session).

## 1. Install and import dependencies

In [ ]:
# Colab already has earthengine-api preinstalled, but geemap needs install.
!pip install -q geemap geopandas


In [ ]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
from pathlib import Path


## 2. Authenticate Earth Engine

First run: a link will open — sign in with your Google account that has Earth Engine access, paste the token back.

If you already have an EE project, pass its ID to `ee.Initialize(project=...)`. If not, you can create one at https://code.earthengine.google.com/

In [ ]:
# Replace 'your-ee-project-id' with your actual project id (e.g., 'ee-cantekin-drought')
EE_PROJECT = 'your-ee-project-id'

try:
    ee.Initialize(project=EE_PROJECT)
    print('Earth Engine initialized successfully.')
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print('Earth Engine authenticated and initialized.')


## 3. Upload Thrace shapefile

Upload all four files together: `.shp`, `.shx`, `.dbf`, `.prj`.

Run the cell below and use the file picker — select all four files at once (Ctrl/Cmd+click).

In [ ]:
from google.colab import files
uploaded = files.upload()

# Find the .shp filename
shp_files = [f for f in uploaded.keys() if f.endswith('.shp')]
if not shp_files:
    raise FileNotFoundError('No .shp file uploaded. Please upload .shp + .shx + .dbf + .prj together.')
SHP_PATH = shp_files[0]
print(f'Shapefile detected: {SHP_PATH}')


## 4. Load and inspect the shapefile

Since the shapefile covers Thrace + European side of Istanbul, we'll use the full extent as our AOI. No cropping needed — Sarıyer station falls within this area.

In [ ]:
gdf = gpd.read_file(SHP_PATH)

print('CRS:', gdf.crs)
print('Number of features:', len(gdf))
print('Bounds:', gdf.total_bounds)
print()
print('First few rows:')
gdf.head()


In [ ]:
# Reproject to WGS84 (EPSG:4326) — required for Earth Engine
if gdf.crs is None:
    print('WARNING: CRS is missing. Assuming EPSG:4326 (WGS84).')
    gdf = gdf.set_crs('EPSG:4326')
elif gdf.crs.to_epsg() != 4326:
    print(f'Reprojecting from {gdf.crs} to EPSG:4326...')
    gdf = gdf.to_crs('EPSG:4326')

print('Final CRS:', gdf.crs)
print('Bounds (WGS84):', gdf.total_bounds)


In [ ]:
# Save a clean GeoJSON for the repo (data/thrace_boundary.geojson)
Path('data').mkdir(exist_ok=True)
GEOJSON_PATH = 'data/thrace_boundary.geojson'

# Dissolve all features into a single polygon for simpler AOI handling
aoi_dissolved = gdf.dissolve()
aoi_dissolved.to_file(GEOJSON_PATH, driver='GeoJSON')

print(f'Saved: {GEOJSON_PATH}')


## 5. Convert AOI to Earth Engine geometry

In [ ]:
# geemap utility handles the conversion cleanly
aoi_ee = geemap.geopandas_to_ee(aoi_dissolved)
aoi_geom = aoi_ee.geometry()

# Sanity check: area in km²
area_km2 = aoi_geom.area().divide(1e6).getInfo()
print(f'AOI area: {area_km2:,.0f} km²')


## 6. Define the 8 stations from the Scientific Reports manuscript

These coordinates are approximate — please replace with the exact coordinates from your MGM/TAGEM metadata before relying on them for validation.

In [ ]:
# Approximate MGM station coordinates — REPLACE with exact values from your station metadata
stations = pd.DataFrame([
    {'name': 'Edirne',      'lat': 41.6667, 'lon': 26.5667, 'elevation_m':  51},
    {'name': 'Kirklareli',  'lat': 41.7333, 'lon': 27.2167, 'elevation_m': 232},
    {'name': 'Luleburgaz',  'lat': 41.4000, 'lon': 27.3500, 'elevation_m':  46},
    {'name': 'Uzunkopru',   'lat': 41.2667, 'lon': 26.6833, 'elevation_m':  22},
    {'name': 'Ipsala',      'lat': 40.9167, 'lon': 26.3833, 'elevation_m':  10},
    {'name': 'Corlu',       'lat': 41.1500, 'lon': 27.8000, 'elevation_m': 183},
    {'name': 'Tekirdag',    'lat': 40.9833, 'lon': 27.5167, 'elevation_m':   3},
    {'name': 'Sariyer',     'lat': 41.1667, 'lon': 29.0500, 'elevation_m':  59},
])

stations.to_csv('data/thrace_stations.csv', index=False)
print('Saved: data/thrace_stations.csv')
stations


In [ ]:
# Convert stations to Earth Engine FeatureCollection
station_features = []
for _, row in stations.iterrows():
    pt = ee.Geometry.Point([row['lon'], row['lat']])
    station_features.append(ee.Feature(pt, {'name': row['name'], 'elevation_m': int(row['elevation_m'])}))

stations_ee = ee.FeatureCollection(station_features)
print('Stations loaded into Earth Engine:', stations_ee.size().getInfo())


## 7. Select a clear Landsat 9 scene over the AOI

Using Landsat 9 Collection 2 Level 2 (surface reflectance + surface temperature, atmospherically corrected). Filtering for scenes with <20% cloud cover during summer 2023 for our MVP demo.

In [ ]:
l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
      .filterBounds(aoi_geom)
      .filterDate('2023-06-01', '2023-09-30')
      .filter(ee.Filter.lt('CLOUD_COVER', 20))
      .sort('CLOUD_COVER'))

n_scenes = l9.size().getInfo()
print(f'Clear scenes found: {n_scenes}')

# Take the single clearest scene
scene = ee.Image(l9.first())
scene_id = scene.get('LANDSAT_PRODUCT_ID').getInfo()
scene_date = ee.Date(scene.get('system:time_start')).format('YYYY-MM-dd').getInfo()
cloud_cover = scene.get('CLOUD_COVER').getInfo()
print(f'Selected scene: {scene_id}')
print(f'Date: {scene_date} | Cloud cover: {cloud_cover}%')


## 8. Apply scaling factors and compute LST

Landsat C2 L2 uses scale factors for both surface reflectance and surface temperature. ST_B10 scale is `0.00341802 * DN + 149` giving Kelvin.

In [ ]:
def apply_scale_factors(image):
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermal_k = image.select('ST_B10').multiply(0.00341802).add(149.0)
    thermal_c = thermal_k.subtract(273.15).rename('LST_C')
    return image.addBands(optical, None, True).addBands(thermal_k.rename('LST_K')).addBands(thermal_c)

scene_scaled = apply_scale_factors(scene).clip(aoi_geom)
print('Scaling applied. Bands:', scene_scaled.bandNames().getInfo()[:15], '...')


## 9. Build the interactive map — THE OUTPUT OF DAY 1

In [ ]:
m = geemap.Map(center=[41.3, 27.2], zoom=8)
m.add_basemap('SATELLITE')

# Landsat 9 true-color RGB
rgb_vis = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0.0, 'max': 0.3}
m.addLayer(scene_scaled, rgb_vis, f'Landsat 9 RGB ({scene_date})')

# Land Surface Temperature (°C)
lst_vis = {'bands': ['LST_C'], 'min': 15, 'max': 45,
           'palette': ['040274', '0502ff', '5dfff0', 'ffc900', 'ff0000', '800080']}
m.addLayer(scene_scaled.select('LST_C'), lst_vis, f'LST °C ({scene_date})', False)

# AOI boundary
m.addLayer(aoi_ee, {'color': 'yellow'}, 'Thrace AOI')

# Stations
m.addLayer(stations_ee, {'color': 'red'}, '8 stations (Scientific Reports manuscript)')

m.addLayerControl()
m


## 10. Day 1 deliverables checklist

If everything ran successfully, you now have:
- `data/thrace_boundary.geojson` — clean, dissolved AOI in WGS84
- `data/thrace_stations.csv` — 8 stations matching the Scientific Reports manuscript
- An interactive map showing AOI + stations + one clear Landsat 9 scene with LST
- A reproducible notebook committable to the repo

**Next (Day 2):**
- Download ERA5-Land meteorology for the scene date
- Compute net radiation (Rn) over the AOI
- Compute instantaneous albedo from scaled reflectance

**Before committing:** Download the two files from Colab (`data/thrace_boundary.geojson`, `data/thrace_stations.csv`) and replace the approximate station coordinates with exact MGM values.